In [2]:
 !pip install ultralytics kagglehub scikit-learn -q

import os
import shutil
import kagglehub
import numpy as np
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

# =============================================================================
# 1) BAIXAR OS BANCOS DE DADOS ONLINE (DIRETO NO SERVIDOR DO COLAB)
# =============================================================================
# Banco 1: LIDC-IDRI (fatias com nódulos anotados por 4 radiologistas)
path_lidc = kagglehub.dataset_download("zhangweiled/lidcidri")
RAIZ_LIDC = Path(path_lidc) / "LIDC-IDRI-slices"
if not RAIZ_LIDC.exists():
    RAIZ_LIDC = Path("/root/.cache/kagglehub/datasets/zhangweiled/lidcidri/versions/1/LIDC-IDRI-slices")

# Bancos 2 e 3: Tomografias de Pulmão Saudável / Normal (sem nódulo)
path_normal_1 = kagglehub.dataset_download("hamdallak/the-iqothnccd-lung-cancer-dataset")
path_normal_2 = kagglehub.dataset_download("mohamedhanyyy/chest-ctscan-images")

SAIDA = Path("/content/data_lidc/yolo")
if SAIDA.exists():
    shutil.rmtree(SAIDA)

for split in ("train", "val"):
    (SAIDA / "images" / split).mkdir(parents=True, exist_ok=True)
    (SAIDA / "labels" / split).mkdir(parents=True, exist_ok=True)

# =============================================================================
# 2) SEPARAR PACIENTES DO LIDC-IDRI EM TREINO (85%) E VALIDAÇÃO (15%)
# =============================================================================
pacientes = sorted([p for p in os.listdir(RAIZ_LIDC) if (RAIZ_LIDC / p).is_dir()])
pacientes_train, pacientes_val = train_test_split(pacientes, test_size=0.15, random_state=42)

split_de = {}
for p in pacientes_train: split_de[p] = "train"
for p in pacientes_val:   split_de[p] = "val"

print(f"Pacientes LIDC no treino: {len(pacientes_train)} | na validação: {len(pacientes_val)}")

# =============================================================================
# 3) PROCESSAR FATIAS COM NÓDULO (LIDC-IDRI)
# =============================================================================
n_ok, n_vazia = 0, 0

for paciente in pacientes:
    split = split_de[paciente]
    pasta_paciente = RAIZ_LIDC / paciente

    for nodulo in sorted(os.listdir(pasta_paciente)):
        pasta_nodulo = pasta_paciente / nodulo
        pasta_imgs = pasta_nodulo / "images"
        if not pasta_imgs.exists():
            continue

        arquivos_img = sorted(os.listdir(pasta_imgs))

        for i, nome_img in enumerate(arquivos_img):
            mascara_combinada = None
            for k in range(4):
                pasta_mask = pasta_nodulo / f"mask-{k}"
                if not pasta_mask.exists():
                    continue
                arquivos_mask = sorted(os.listdir(pasta_mask))
                if i >= len(arquivos_mask):
                    continue
                m = np.array(Image.open(pasta_mask / arquivos_mask[i]).convert("L")) > 127
                mascara_combinada = m if mascara_combinada is None else (mascara_combinada | m)

            if mascara_combinada is None or mascara_combinada.sum() == 0:
                n_vazia += 1
                continue  # pula as 62 fatias com erro de máscara vazia

            ys, xs = np.where(mascara_combinada)
            x1, x2 = int(xs.min()), int(xs.max())
            y1, y2 = int(ys.min()), int(ys.max())

            with Image.open(pasta_imgs / nome_img) as im:
                img_w, img_h = im.size

            x_center = ((x1 + x2) / 2) / img_w
            y_center = ((y1 + y2) / 2) / img_h
            w = (x2 - x1) / img_w
            h = (y2 - y1) / img_h

            if w <= 0 or h <= 0:
                continue

            nome_saida = f"{paciente}_{nodulo}_{i:03d}.png"
            shutil.copy2(pasta_imgs / nome_img, SAIDA / "images" / split / nome_saida)

            label_path = SAIDA / "labels" / split / (Path(nome_saida).stem + ".txt")
            label_path.write_text(f"0 {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")

            n_ok += 1

print(f"Fatias COM nódulo processadas: {n_ok} (puladas: {n_vazia})")

# =============================================================================
# 4) ADICIONAR IMAGENS DE PULMÃO SAUDÁVEL / SEM NÓDULO (.txt EM BRANCO)
# =============================================================================
imagens_normais = []
for base_normal in (path_normal_1, path_normal_2):
    for raiz_dir, _, arquivos in os.walk(base_normal):
        if "normal" in Path(raiz_dir).name.lower():
            for arq in sorted(arquivos):
                if arq.lower().endswith((".png", ".jpg", ".jpeg")):
                    imagens_normais.append(Path(raiz_dir) / arq)

norm_train, norm_val = train_test_split(imagens_normais, test_size=0.15, random_state=42)

for split, lista_imgs in (("train", norm_train), ("val", norm_val)):
    for idx, caminho_img in enumerate(lista_imgs):
        nome_saida = f"normal_saudavel_{idx:04d}.png"
        destino_img = SAIDA / "images" / split / nome_saida
        destino_lbl = SAIDA / "labels" / split / f"normal_saudavel_{idx:04d}.txt"

        # Converte para escala de cinza (L) igual ao LIDC-IDRI e salva como PNG
        with Image.open(caminho_img) as im:
            im.convert("L").save(destino_img)

        # Arquivo .txt em branco indica ao YOLO que não há nódulo na imagem
        destino_lbl.write_text("")

print(f"Imagens de pulmão saudável adicionadas: {len(imagens_normais)} (Treino: {len(norm_train)} | Val: {len(norm_val)})")

# =============================================================================
# 5) CRIAR dataset.yaml E TREINAR O MODELO YOLOv8s
# =============================================================================
yaml_content = """path: /content/data_lidc/yolo
train: images/train
val: images/val
names:
  0: nodulo
"""

with open("/content/data_lidc/yolo/dataset.yaml", "w") as f:
    f.write(yaml_content)

model_lidc = YOLO("yolov8s.pt")
model_lidc.train(
    data="/content/data_lidc/yolo/dataset.yaml",
    epochs=50,
    imgsz=224,
    batch=32,
    patience=15,
    name="lidc_paciente_com_saudaveis",
    optimizer="AdamW"
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 13.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


100%|██████████| 163M/163M [00:04<00:00, 38.0MB/s]

Extracting files...


Using Colab cache for faster access to the 'the-iqothnccd-lung-cancer-dataset' dataset.


100%|██████████| 119M/119M [00:03<00:00, 35.2MB/s]

Extracting files...


Pacientes LIDC no treino: 743 | na validação: 132
Fatias COM nódulo processadas: 15362 (puladas: 62)
Imagens de pulmão saudável adicionadas: 631 (Treino: 536 | Val: 95)
Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_lidc/yolo/dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, kobj=1.0, lin

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a02a58a2ac0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [3]:
import os
import shutil
from pathlib import Path
from google.colab import files, drive

PASTA_TREINO = Path("/content/runs/detect/lidc_paciente_com_saudaveis")
ARQUIVO_BEST = PASTA_TREINO / "weights" / "best.pt"
ARQUIVO_ZIP = "/content/resultado_completo_treino_lidc"

if not ARQUIVO_BEST.exists():
    raise FileNotFoundError(f"❌ Não encontrei o arquivo em {ARQUIVO_BEST}. Verifique se a 1ª célula terminou de rodar!")

# =============================================================================
# 1) COMPACTAR TUDO E BAIXAR DIRETO PARA O SEU PC (PASTA DOWNLOADS)
# =============================================================================
print("📦 Compactando pesos, gráficos e métricas (results.csv)...")
caminho_zip_gerado = shutil.make_archive(ARQUIVO_ZIP, "zip", PASTA_TREINO)

print("⬇️ Iniciando download do best.pt e do ZIP completo para o seu PC...")
files.download(str(ARQUIVO_BEST))
files.download(caminho_zip_gerado)

# =============================================================================
# 2) SALVAR CÓPIA DE SEGURANÇA NO SEU GOOGLE DRIVE
# =============================================================================
print("\n☁️ Conectando ao Google Drive para salvar cópia de segurança...")
drive.mount("/content/drive", force_remount=True)

# Trava de segurança para garantir que o Drive montou de verdade
if not os.path.ismount("/content/drive"):
    raise RuntimeError("⚠️ O Google Drive não foi montado! Aceite a permissão na janela do Colab e rode esta célula de novo.")

PASTA_DRIVE = Path("/content/drive/MyDrive/TCC_IA_NODULOS_LIDC")
PASTA_DRIVE.mkdir(parents=True, exist_ok=True)

# Copia o best.pt e o zip completo para o Drive
shutil.copy2(ARQUIVO_BEST, PASTA_DRIVE / "best_paciente_com_saudaveis.pt")
shutil.copy2(caminho_zip_gerado, PASTA_DRIVE / "resultado_completo_treino_lidc.zip")

print(f"✅ TUDO SALVO COM SUCESSO NO GOOGLE DRIVE EM: {PASTA_DRIVE}")
print("Arquivos salvos no Drive:", os.listdir(PASTA_DRIVE))

📦 Compactando pesos, gráficos e métricas (results.csv)...
⬇️ Iniciando download do best.pt e do ZIP completo para o seu PC...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


☁️ Conectando ao Google Drive para salvar cópia de segurança...
Mounted at /content/drive
✅ TUDO SALVO COM SUCESSO NO GOOGLE DRIVE EM: /content/drive/MyDrive/TCC_IA_NODULOS_LIDC
Arquivos salvos no Drive: ['best_paciente_com_saudaveis.pt', 'resultado_completo_treino_lidc.zip']
